# Grow-and-compare: K=1 vs K=10 (LeiData WAV 1)

Latest run (built when the closeness knob was still a percentile,
`PCA_DISTANCE_PERCENTILE=50`; new runs use `MAX_RELATIVE_DISTANCE`):

| Setting | Value |
|---|---|
| Windows | **30** (full wav1 catalog) |
| Templates requested | 10 each |
| `n_components` | 1 vs 10 |
| Seed policy | `low_coverage` (match on full stack) |
| PCA distance percentile | **50** (closer half for PCA covariance) |
| `eps_step_fraction` | 0.1 |
| Refine iters | 2 |

Loads manifests from `wav1_k1_k10_grow_compare/` (latest stamps). Does not re-learn.

```bash
MAX_RELATIVE_DISTANCE=0.9 bash analysis/dict_learning/run_compare_grow_k1_k10_leidata_wav1.sh
```



In [ ]:
%matplotlib inline

import json
import sys
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REPO = Path("/home/yfreund/projects/gplrev3-webserver")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

LIBRARY_ROOT = Path("/home/yfreund/whale_processing")
COMPARE_ROOT = Path(
    LIBRARY_ROOT / "result/LeiData/dict_learning_templates/wav1_k1_k10_grow_compare"
)
DB_PATH = LIBRARY_ROOT / "result/dict_learning_catalog.sqlite"
MIN_TEMPLATES = 5
# Spectrogram overlays: show this many windows (evenly spaced); energy uses all.
MAX_OVERLAY_WINDOWS = 30  # show all windows (K=1 and K=10 alternate → 60 panels)


def _latest_stamp(parent: Path) -> Path:
    """Newest stamped run that has a non-empty manifest.json."""
    stamps = sorted(
        p for p in parent.iterdir()
        if p.is_dir() and (p / "manifest.json").is_file()
        and (p / "manifest.json").stat().st_size > 0
    )
    if not stamps:
        raise FileNotFoundError(f"No complete stamped runs under {parent}")
    return stamps[-1]


def _run_row_from_manifest(out_dir: Path, n_components: int) -> dict:
    man = json.loads((out_dir / "manifest.json").read_text())
    e0 = float(man["energy_before"])
    e1 = float(man["energy_after"])
    removed = e0 - e1
    frac = removed / e0 if e0 else 0.0
    params = man.get("params") or {}
    return {
        "n_components": int(man.get("n_components", n_components)),
        "out_dir": str(out_dir),
        "run_stamp": out_dir.name,
        "dictionary_id": None,
        "n_templates": len(man["templates"]),
        "n_matchings": len(man["matchings"]),
        "n_windows": len(man.get("window_ids") or []),
        "stop_reason": man.get("stop_reason"),
        "energy_before": e0,
        "energy_after": e1,
        "energy_removed": removed,
        "fraction_removed": frac,
        "percent_removed": 100.0 * frac,
        "refine_iters": int(params.get("refine_iters", 0)),
        "seed_window_policy": params.get("seed_window_policy"),
        "eps_step_fraction": params.get("eps_step_fraction"),
        # Runs before the relative-distance gate recorded a percentile instead.
        "closeness_knob": (
            f"max_relative_distance={params['max_relative_distance']}"
            if params.get("max_relative_distance") is not None
            else f"pca_distance_percentile={params.get('pca_distance_percentile')}"
        ),
    }


def _discover_ks(compare_root: Path, summary: dict | None = None) -> list[int]:
    if summary and summary.get("ks"):
        return [int(k) for k in summary["ks"]]
    if summary and summary.get("runs"):
        ks = sorted(int(str(k).lstrip("k")) for k in summary["runs"])
        if len(ks) == 2:
            return ks
    found = []
    for p in sorted(compare_root.glob("wav1_grow_k*")):
        if p.is_dir():
            found.append(int(p.name.split("wav1_grow_k", 1)[1]))
    if len(found) != 2:
        raise FileNotFoundError(
            f"Expected exactly two wav1_grow_k* dirs under {compare_root}, got {found}"
        )
    return found


def load_or_rebuild_summary(compare_root: Path) -> dict:
    """Rebuild summary from the latest complete wav1_grow_k* stamps."""
    summary_path = compare_root / "compare_summary.json"
    existing = None
    if summary_path.is_file():
        existing = json.loads(summary_path.read_text())
    ks = _discover_ks(compare_root, existing)
    runs = {}
    for k in ks:
        stamp = _latest_stamp(compare_root / f"wav1_grow_k{k}")
        runs[f"k{k}"] = _run_row_from_manifest(stamp, k)
        print(f"  k{k}: {stamp}")
    low, high = ks
    # Prefer policy/eps from compare_summary, else from a run manifest.
    policy = (existing or {}).get("seed_window_policy") or runs[f"k{low}"].get(
        "seed_window_policy"
    )
    eps = (existing or {}).get("eps_step_fraction") or runs[f"k{low}"].get(
        "eps_step_fraction"
    )
    neighborhood = (existing or {}).get("seed_window_neighborhood")
    closeness = (existing or {}).get("closeness_knob") or runs[f"k{low}"].get(
        "closeness_knob"
    )
    summary = {
        "n_windows": max(r["n_windows"] for r in runs.values()),
        "n_templates_requested": (existing or {}).get("n_templates_requested"),
        "refine_iters": max(r["refine_iters"] for r in runs.values()),
        "min_templates": MIN_TEMPLATES,
        "ks": ks,
        "seed_window_policy": policy,
        "seed_window_neighborhood": neighborhood,
        "eps_step_fraction": eps,
        "closeness_knob": closeness,
        "runs": runs,
        "delta_percent_removed": (
            runs[f"k{high}"]["percent_removed"] - runs[f"k{low}"]["percent_removed"]
        ),
        "rebuilt_from_manifests": True,
    }
    summary_path.write_text(json.dumps(summary, indent=2))
    summary["summary_path"] = str(summary_path)
    print(f"Using latest stamps → {summary_path}")
    return summary


if not COMPARE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Missing compare root {COMPARE_ROOT}\n"
        "Run: bash analysis/dict_learning/run_compare_grow_k1_k10_leidata_wav1.sh"
    )

summary = load_or_rebuild_summary(COMPARE_ROOT)
KS = [int(k) for k in summary["ks"]]
RUN_KEYS = [f"k{k}" for k in KS]
DISPLAY_LABELS = [f"K={k}" for k in KS]
print(f"Compare root: {COMPARE_ROOT}")
print(
    f"ks={KS}  n_windows={summary.get('n_windows')}  "
    f"n_templates_requested={summary.get('n_templates_requested')}  "
    f"refine_iters={summary.get('refine_iters')}  "
    f"seed_window_policy={summary.get('seed_window_policy')}  "
    f"{summary.get('closeness_knob')}  "
    f"eps_step_fraction={summary.get('eps_step_fraction')}"
)
for key, label in zip(RUN_KEYS, DISPLAY_LABELS):
    r = summary["runs"][key]
    print(
        f"  {label}: templates={r['n_templates']} matchings={r['n_matchings']} "
        f"E {r['energy_before']:.6g} → {r['energy_after']:.6g} "
        f"({r['percent_removed']:.2f}% removed)\n"
        f"       out={r['out_dir']}"
    )
print(
    f"Δ percent removed (K={KS[1]} − K={KS[0]}) = "
    f"{summary['delta_percent_removed']:+.2f} pp"
)

if summary.get("n_windows", 0) < 4:
    warnings.warn(f"n_windows={summary.get('n_windows')} < 4")
if (summary.get("refine_iters") or 0) < 2:
    warnings.warn(
        f"refine_iters={summary.get('refine_iters')} < 2 "
        "(re-run the script with REFINE_ITERS=2 if needed)"
    )
for key in RUN_KEYS:
    n = summary["runs"][key]["n_templates"]
    if n < MIN_TEMPLATES:
        warnings.warn(
            f"{key} has only {n} templates (want >= {MIN_TEMPLATES}). "
            "Re-run with larger N_TEMPLATES / more windows / lower min_repeats."
        )


In [ ]:
def load_run(out_dir: str | Path) -> dict:
    out = Path(out_dir)
    man = json.loads((out / "manifest.json").read_text())
    templates = {}
    widths = {}
    actual_ks = {}
    for item in man["templates"]:
        tid = int(item["template_id"])
        arr = np.load(out / item["file"])
        templates[tid] = arr
        widths[tid] = int(arr.shape[-1])
        actual_ks[tid] = int(arr.shape[0]) if arr.ndim == 3 else 1
    return {
        "manifest": man,
        "templates": templates,
        "widths": widths,
        "actual_ks": actual_ks,
        "matchings": man["matchings"],
        "window_ids": list(man.get("window_ids") or []),
        "kind": man.get("kind", "rank1"),
        "n_components": int(man.get("n_components", 1)),
        "energy_before": float(man["energy_before"]),
        "energy_after": float(man["energy_after"]),
    }


runs = {
    label: load_run(summary["runs"][key]["out_dir"])
    for label, key in zip(DISPLAY_LABELS, RUN_KEYS)
}
LOW_LABEL, HIGH_LABEL = DISPLAY_LABELS
for name, run in runs.items():
    actual = sorted(run["actual_ks"].values())
    print(
        f"{name}: kind={run['kind']} n_components={run['n_components']} "
        f"templates={len(run['templates'])} matchings={len(run['matchings'])} "
        f"actual_k_per_atom={actual}"
    )


## Energy reduction


In [ ]:
import pandas as pd

rows = []
for label, key in zip(DISPLAY_LABELS, RUN_KEYS):
    r = summary["runs"][key]
    rows.append({
        "method": label,
        "n_components": r["n_components"],
        "n_templates": r["n_templates"],
        "n_matchings": r["n_matchings"],
        "E0": r["energy_before"],
        "E1": r["energy_after"],
        "E_removed": r["energy_removed"],
        "% removed": r["percent_removed"],
    })
df = pd.DataFrame(rows)
display(df)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), constrained_layout=True)
ax = axes[0]
x = np.arange(len(rows))
ax.bar(x, [r["% removed"] for r in rows], color=["#4C72B0", "#DD8452"])
ax.set_xticks(x)
ax.set_xticklabels([r["method"] for r in rows])
ax.set_ylabel("% residual energy removed")
ax.set_title(
    f"Energy on {summary['n_windows']} windows "
    f"(policy={summary.get('seed_window_policy')})"
)
for i, r in enumerate(rows):
    ax.text(i, r["% removed"] + 0.3, f"{r['% removed']:.2f}%", ha="center")

# Matches per template (balance view)
ax = axes[1]
width = 0.35
for i, (label, key) in enumerate(zip(DISPLAY_LABELS, RUN_KEYS)):
    run = runs[label]
    counts = Counter(int(m["template_id"]) for m in run["matchings"])
    tids = sorted(run["templates"])
    ys = [counts[t] for t in tids]
    ax.bar(np.arange(len(tids)) + i * width, ys, width=width, label=label)
ax.set_xlabel("template id")
ax.set_ylabel("match count")
ax.set_title("Matches per template")
ax.legend()
plt.show()

print(f"Δ % removed = {summary['delta_percent_removed']:+.2f} pp")


## Matching locations on spectrograms

One **panel per row**, alternating low-K then high-K for **all** windows
(`MAX_OVERLAY_WINDOWS=30`). Spectrograms use **jet**; match footprints are
colored outlines with template-id labels.

Note: this produces 60 panels — scroll the output to review each window.


In [ ]:
import sqlite3

# Build the full window list from manifests + matchings.
window_ids = []
for label in DISPLAY_LABELS:
    for wid in (runs[label]["window_ids"] or []):
        if wid not in window_ids:
            window_ids.append(wid)
seen = []
for run in runs.values():
    for m in run["matchings"]:
        wid = m["window_id"]
        if wid not in seen:
            seen.append(wid)
if len(seen) > len(window_ids):
    window_ids = seen

if len(window_ids) < 2:
    raise RuntimeError(
        f"Expected >=4 windows for overlays, got {len(window_ids)}: {window_ids}. "
        "Re-run the first cells after the grow script."
    )

# Evenly spaced sample for display when many windows.
n_all = len(window_ids)
if n_all > MAX_OVERLAY_WINDOWS:
    idxs = np.linspace(0, n_all - 1, MAX_OVERLAY_WINDOWS, dtype=int)
    overlay_ids = [window_ids[i] for i in idxs]
    print(
        f"Overlaying {len(overlay_ids)}/{n_all} windows "
        f"(even sample; set MAX_OVERLAY_WINDOWS to change):"
    )
else:
    overlay_ids = list(window_ids)
    print(
        f"Overlaying {len(overlay_ids)} windows "
        f"({LOW_LABEL} then {HIGH_LABEL}, one panel per row):"
    )
for i, wid in enumerate(overlay_ids):
    print(f"  [{i}] {wid}")


def load_window_array(window_id: str) -> np.ndarray:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT npy_path FROM windows WHERE window_id = ?",
            (window_id,),
        ).fetchone()
        if row is None:
            raise KeyError(f"window_id not in catalog: {window_id}")
        return np.asarray(np.load(Path(row["npy_path"])), dtype=np.float64)


def plot_overlays(ax, run, spectrogram: np.ndarray, window_id: str, title: str) -> int:
    matchings = [m for m in run["matchings"] if m["window_id"] == window_id]
    active_ids = sorted({int(m["template_id"]) for m in matchings})
    color_cycle = plt.cm.tab20(np.linspace(0, 1, max(20, len(active_ids) or 1)))
    colors = {tid: color_cycle[i % len(color_cycle)] for i, tid in enumerate(active_ids)}

    vmin, vmax = np.percentile(spectrogram, [5, 99.5])
    ax.imshow(
        spectrogram,
        aspect="auto",
        origin="lower",
        cmap="jet",
        vmin=vmin,
        vmax=vmax,
    )
    n_freq = spectrogram.shape[0]
    for item in sorted(matchings, key=lambda m: m["t_start"]):
        tid = int(item["template_id"])
        width = run["widths"][tid]
        color = colors[tid]
        amp = float(item.get("amplitude", 0.0))
        ax.add_patch(
            mpatches.Rectangle(
                (item["t_start"] - 0.5, -0.5),
                width,
                n_freq,
                facecolor="none",
                edgecolor=color,
                linewidth=2.0,
                linestyle="-" if amp >= 0 else "--",
            )
        )
        ax.annotate(
            str(tid),
            (item["t_start"] + width / 2, n_freq * 0.97),
            color="white",
            ha="center",
            va="top",
            fontsize=8,
            fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.15", fc=color, ec="none", alpha=0.85),
        )
    ax.set_title(f"{title}: {len(matchings)} matches", fontsize=11)
    ax.set_xlabel("Time bin")
    ax.set_ylabel("Frequency bin")
    return len(matchings)


n_win = len(overlay_ids)
n_rows = 2 * n_win
fig, axes = plt.subplots(
    n_rows, 1, figsize=(14, 3.2 * n_rows), constrained_layout=True, squeeze=False
)

for i, wid in enumerate(overlay_ids):
    spec = load_window_array(wid)
    short = wid if len(wid) < 80 else wid[:36] + "…" + wid[-36:]
    counts = []
    for j, label in enumerate(DISPLAY_LABELS):
        n = plot_overlays(
            axes[2 * i + j, 0],
            runs[label],
            spec,
            wid,
            f"W{i + 1}/{n_all} {label} — {short}",
        )
        counts.append(n)
    print(
        f"Window sample {i + 1}: "
        + "  ".join(f"{lab} matches={n}" for lab, n in zip(DISPLAY_LABELS, counts))
    )

fig.suptitle(
    f"Match footprints — sample of {n_win}/{n_all} windows, "
    f"alternating {LOW_LABEL} / {HIGH_LABEL} (jet)",
    y=1.01,
)
plt.show()


## Template gallery

- **Low-K (usually K=1):** each panel is the rank-1 atom.
- **High-K:** each **row** is one template; columns are **PC1…PCk** (actual stored
  modes; may be `< requested K` when an atom had fewer patches). Shared color
  scale within a row.
  Labels: `t=` template id, `PC=` mode, `m=` match count; `λ=` PCA variance when
  stored in the manifest.


In [ ]:
def show_templates(run, title: str, *, show_all_modes: bool = False):
    """Plot template atoms. For subspace atoms, optionally show every PCA mode."""
    tids = sorted(run["templates"])
    counts = Counter(int(m["template_id"]) for m in run["matchings"])

    if not show_all_modes:
        n = len(tids)
        n_cols = min(5, max(1, n))
        n_rows = int(np.ceil(n / n_cols))
        fig, axes = plt.subplots(
            n_rows, n_cols, figsize=(2.8 * n_cols, 3.0 * n_rows), squeeze=False
        )
        for ax, tid in zip(axes.flat, tids):
            data = run["templates"][tid]
            patch = data[0] if data.ndim == 3 else data
            clim = np.percentile(np.abs(patch), 99) or 1.0
            ax.imshow(patch, aspect="auto", origin="lower", cmap="RdBu_r", vmin=-clim, vmax=clim)
            k = data.shape[0] if data.ndim == 3 else 1
            ax.set_title(f"t={tid} k={k} m={counts[tid]}")
            ax.set_xticks([])
            ax.set_yticks([])
        for ax in axes.flat[n:]:
            ax.axis("off")
        fig.suptitle(title, y=1.02)
        plt.tight_layout()
        plt.show()
        return

    # One row per template; columns = PCA modes (PC1, PC2, …).
    max_k = max(
        (arr.shape[0] if arr.ndim == 3 else 1) for arr in run["templates"].values()
    )
    panel_w = 2.2 if max_k <= 4 else 1.6
    fig, axes = plt.subplots(
        len(tids),
        max_k,
        figsize=(panel_w * max_k, 2.4 * max(1, len(tids))),
        squeeze=False,
    )
    for row, tid in enumerate(tids):
        data = run["templates"][tid]
        modes = [data] if data.ndim == 2 else [data[m] for m in range(data.shape[0])]
        clim = max((np.percentile(np.abs(p), 99) for p in modes), default=1.0) or 1.0
        variances = None
        for item in run["manifest"]["templates"]:
            if int(item["template_id"]) == tid and "variances" in item:
                variances = item["variances"]
                break
        for col in range(max_k):
            ax = axes[row, col]
            if col >= len(modes):
                ax.axis("off")
                continue
            ax.imshow(
                modes[col],
                aspect="auto",
                origin="lower",
                cmap="RdBu_r",
                vmin=-clim,
                vmax=clim,
            )
            subtitle = f"t={tid} PC{col + 1}  m={counts[tid]}"
            if variances is not None and col < len(variances):
                subtitle += f"\nλ={variances[col]:.3g}"
            ax.set_title(subtitle, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(f"t={tid} (k={len(modes)})", fontsize=9)
    fig.suptitle(title, y=1.01)
    plt.tight_layout()
    plt.show()


show_templates(runs[LOW_LABEL], f"{LOW_LABEL} templates")
show_templates(
    runs[HIGH_LABEL],
    f"{HIGH_LABEL} templates — PCA modes per atom (actual k may be < requested)",
    show_all_modes=True,
)
